# SUPIR с явным циклом сэмплинга

Минимальный вариант без LLaVA. Сеть, кодирование, декодирование, сетка уровней шума и CFG берутся из репозитория; **цикл EDM написан ниже в ноутбуке**. Изменять его можно в `modify_prediction(...)` или непосредственно в `sample_latent(...)`. Веса и CUDA-окружение готовятся так же, как в [базовом ноутбуке](supir_baseline.ipynb). Полный инференс здесь заранее не выполнялся.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'SUPIR').is_dir(), 'Запустите ноутбук из корня репозитория или notebooks/'
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
from PIL import Image
from omegaconf import OmegaConf
from pytorch_lightning import seed_everything
import CKPT_PTH

CKPT_PTH.SDXL_CLIP1_PATH = None  # локальный путь, если нет доступа к Hugging Face
CKPT_PTH.SDXL_CLIP2_CKPT_PTH = None
from SUPIR.util import create_SUPIR_model, PIL2Tensor, Tensor2PIL, convert_dtype
from sgm.util import instantiate_from_config
from SUPIR.utils.colorfix import wavelet_reconstruction

assert torch.cuda.is_available(), 'Эта версия SUPIR требует CUDA'
DEVICE = 'cuda:0'

## Модель и параметры

Пути к двум основным весам и входному изображению можно поменять. Остальные параметры соответствуют базовому запуску `test.py`; `RESTORATION_SCALE = -1` отключает дополнительное притяжение к LQ-латенту, а положительное значение (например, `4.0`) включает его.

In [ ]:
SDXL_CKPT = ROOT / 'checkpoints/sd_xl_base_1.0_0.9vae.safetensors'
SUPIR_CKPT = ROOT / 'checkpoints/SUPIR-v0Q.ckpt'
INPUT_IMAGE = ROOT / 'examples/input.png'
STEPS = 50
SEED = 1234
RESTORATION_SCALE = -1.0
CFG_SCALE = 4.0
CFG_SCALE_START = 1.0
CONTROL_SCALE = 1.0
S_CHURN = 5.0
S_NOISE = 1.01
PROMPT = ''  # описание содержимого вручную; качественные слова есть в конфигурации

for path in (SDXL_CKPT, SUPIR_CKPT):
    if not path.is_file():
        raise FileNotFoundError(path)

config = OmegaConf.load(ROOT / 'options/SUPIR_v0.yaml')
config.SDXL_CKPT = str(SDXL_CKPT)
config.SUPIR_CKPT = None
config.SUPIR_CKPT_Q = str(SUPIR_CKPT)
with tempfile.TemporaryDirectory() as tmpdir:
    config_path = Path(tmpdir) / 'supir_notebook.yaml'
    OmegaConf.save(config, config_path)
    model = create_SUPIR_model(str(config_path), SUPIR_sign='Q')

model.ae_dtype = convert_dtype('bf16')
model.model.dtype = convert_dtype('fp16')
model = model.to(DEVICE).eval()

## Явный цикл

`modify_prediction` получает результат модели **после CFG и стандартной коррекции SUPIR**, перед вычислением направления EDM. По умолчанию она ничего не меняет. Для эксперимента с маской изменяйте эту функцию; возвращайте тензор той же формы, что `prediction`. Параметр `center` — латент входа после первой стадии восстановления.

In [ ]:
def modify_prediction(prediction, *, step, sigma, x, center):
    # Точка расширения для маски или другой коррекции сэмплинга.
    return prediction


@torch.inference_mode()
def sample_latent(model, lq, prompt='', edit=modify_prediction):
    seed_everything(SEED)
    z_control = model.encode_first_stage_with_denoise(lq, use_sample=False)
    stage1 = model.decode_first_stage(z_control)
    center = model.encode_first_stage(stage1)
    cond, uncond = model.prepare_condition(z_control, [prompt], model.p_p, model.n_p, len(lq))

    sampler_params = model.sampler_config.params
    discretization = instantiate_from_config(sampler_params.discretization_config)
    guider_config = OmegaConf.create(OmegaConf.to_container(sampler_params.guider_config, resolve=True))
    guider_config.params.scale = CFG_SCALE_START
    guider_config.params.scale_min = CFG_SCALE
    guider = instantiate_from_config(guider_config)
    sigmas = discretization(STEPS, device=lq.device)  # убывают; последний элемент равен 0

    x = torch.randn_like(z_control) * torch.sqrt(1 + sigmas[0] ** 2)
    sigma_max = 14.6146  # значение из RestoreEDMSampler этого репозитория
    for step in range(len(sigmas) - 1):
        sigma = sigmas[step].expand(len(x))
        next_sigma = sigmas[step + 1].expand(len(x))
        gamma = min(S_CHURN / (len(sigmas) - 1), 2 ** 0.5 - 1)
        sigma_hat = sigma * (1 + gamma)
        if gamma > 0:
            x = x + torch.randn_like(x) * S_NOISE * (sigma_hat**2 - sigma**2).sqrt()[:, None, None, None]

        model_inputs = guider.prepare_inputs(x, sigma_hat, cond, uncond)
        raw = model.denoiser(model.model, *model_inputs, CONTROL_SCALE)
        prediction = guider(raw, sigma_hat)
        if RESTORATION_SCALE > 0 and next_sigma[0] > 0.05:
            weight = (sigma / sigma_max) ** RESTORATION_SCALE
            prediction = prediction - (prediction - center) * weight[:, None, None, None]

        prediction = edit(prediction, step=step, sigma=sigma, x=x, center=center)
        assert prediction.shape == x.shape, 'modify_prediction должна сохранять форму латента'
        derivative = (x - prediction) / sigma_hat[:, None, None, None]
        x = x + (next_sigma - sigma_hat)[:, None, None, None] * derivative

    restored = model.decode_first_stage(x)
    restored = wavelet_reconstruction(restored, stage1)
    return restored

## Запуск на одном изображении

Это отдельная последняя ячейка: её не нужно выполнять для изучения и изменения алгоритма выше. Результат сохраняется в исключённой из Git папке `results/`.

In [ ]:
if not INPUT_IMAGE.is_file():
    raise FileNotFoundError(INPUT_IMAGE)
image = Image.open(INPUT_IMAGE).convert('RGB')
lq, original_h, original_w = PIL2Tensor(image)
lq = lq.unsqueeze(0).to(DEVICE)
result_tensor = sample_latent(model, lq, prompt=PROMPT)
result = Tensor2PIL(result_tensor[0], original_h, original_w)
output_path = ROOT / 'results/supir_custom_sampling.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
result.save(output_path)
print(output_path)
display(result)